### Emotion Model Training (GoEmotions)
This notebook trains the emotion classification model using vectorized text features from the GoEmotions dataset. The multi-label problem is simplified into a single dominant emotion to enable standard classification.

The trained model is responsible for detecting emotional states from user input, forming the first stage of the HDT behavioral simulation pipeline.

In [1]:
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, hamming_loss

In [2]:
text_features = joblib.load("../data/goemotions_tfidf_features.pkl")
emotion_labels = np.load("../data/goemotions_labels.npy")
print("Features Shape")
print(text_features.shape)
print("Labels Shape")
print(emotion_labels.shape)

Features Shape
(207805, 10000)
Labels Shape
(207805, 28)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(text_features, emotion_labels, test_size = 0.2, random_state = 42)
print("Train Shape")
print(X_train.shape)
print("Test Shape")
print(X_test.shape)

Train Shape
(166244, 10000)
Test Shape
(41561, 10000)


In [4]:
base_classifier = LogisticRegression(solver = 'liblinear', class_weight = 'balanced', max_iter = 2000)
multilabel_model = OneVsRestClassifier(base_classifier)
print("Model Initialized Successfully")

Model Initialized Successfully


In [5]:
multilabel_model.fit(X_train, y_train)
print("GoEmotions Multi-Label Model Trained Successfully")

GoEmotions Multi-Label Model Trained Successfully


In [6]:
y_prob = multilabel_model.predict_proba(X_test)
threshold = 0.65
y_pred = (y_prob >= threshold).astype(int)

In [7]:
micro_f1 = f1_score(y_test, y_pred, average = 'micro')
macro_f1 = f1_score(y_test, y_pred, average = 'macro')
hamming = hamming_loss(y_test, y_pred)
print("Micro F1 Score")
print(micro_f1)
print("Macro F1 Score")
print(macro_f1)
print("Hamming Loss")
print(hamming)

Micro F1 Score
0.31626588353451945
Macro F1 Score
0.2894798569334954
Hamming Loss
0.09243126282538232


In [8]:
joblib.dump(multilabel_model, "../models/goemotions_model.pkl")
print("Model Saved Successfully")

Model Saved Successfully
